# Introduction to Human Language Technologies
## Lab 5: Lexical Semantics



In [1]:
from functools import lru_cache
from itertools import combinations

import nltk
from nltk.corpus import wordnet as wn, wordnet_ic

for resource in ('wordnet', 'wordnet_ic'):
    try:
        nltk.data.find(f'corpora/{resource}')
    except LookupError:
        try:
            nltk.data.find(f'corpora/{resource}.zip')
        except LookupError:
            if not nltk.download(resource, quiet=True):
                raise RuntimeError(f'Could not download {resource}')

brown_ic = wordnet_ic.ic('ic-brown.dat')

Data Preparation

In [2]:
sentences = [
    [('the', 'DT'), ('man', 'NN'), ('swim', 'VB'), ('with', 'PR'),
     ('a', 'DT'), ('girl', 'NN'), ('and', 'CC'), ('a', 'DT'),
     ('boy', 'NN'), ('whilst', 'PR'), ('the', 'DT'), ('woman', 'NN'),
     ('walk', 'VB')],
    [('the', 'DT'), ('dog', 'NN'), ('run', 'VB'), ('after', 'PR'),
     ('a', 'DT'), ('cat', 'NN')]
]

pos_map = {'N': wn.NOUN, 'V': wn.VERB, 'J': wn.ADJ, 'R': wn.ADV}

@lru_cache(maxsize=None)
def most_frequent_synset(lemma, tag):
    pos = pos_map.get(tag[0])
    if pos is None:
        return None
    synsets = wn.synsets(lemma, pos=pos)
    return synsets[0] if synsets else None

Most frequent synsets

In [3]:
for number, sentence in enumerate(sentences, start=1):
    print('\nSentence', number)
    for lemma, tag in sentence:
        synset = most_frequent_synset(lemma, tag)
        name = synset.name() if synset is not None else 'N/A'
        print(lemma, tag, name)

content = {}
for sentence in sentences:
    for lemma, tag in sentence:
        synset = most_frequent_synset(lemma, tag)
        if synset is not None:
            content[(lemma, tag)] = synset


Sentence 1
the DT N/A
man NN man.n.01
swim VB swim.v.01
with PR N/A
a DT N/A
girl NN girl.n.01
and CC N/A
a DT N/A
boy NN male_child.n.01
whilst PR N/A
the DT N/A
woman NN woman.n.01
walk VB walk.v.01

Sentence 2
the DT N/A
dog NN dog.n.01
run VB run.v.01
after PR N/A
a DT N/A
cat NN cat.n.01


LCS and Similarities


In [4]:
lch_max = {}
for synset in content.values():
    if synset.pos() not in lch_max:
        lch_max[synset.pos()] = synset.lch_similarity(synset)

def compare_synsets(first, second):
    if first.pos() != second.pos() or first.pos() not in ('n', 'v'):
        return None
    lcs = sorted(s.name() for s in first.lowest_common_hypernyms(second))
    lch = first.lch_similarity(second)
    return {
        'lcs': lcs,
        'path': first.path_similarity(second),
        'lch': lch,
        'lch_norm': lch / lch_max[first.pos()] if lch is not None else None,
        'wup': first.wup_similarity(second),
        'lin': first.lin_similarity(second, brown_ic)
    }

results = []
skipped = 0
for (word1, synset1), (word2, synset2) in combinations(content.items(), 2):
    scores = compare_synsets(synset1, synset2)
    if scores is None:
        skipped += 1
        continue
    results.append({'words': (word1[0], word2[0]), **scores})

In [5]:
print(len(content), 'content words')
print(len(results), 'comparable pairs')
print(skipped, 'noun/verb pairs skipped')

for result in results:
    print()
    print(*result['words'])
    print('LCS:', result['lcs'])
    print('Path:', result['path'])
    print('LCH:', result['lch'])
    print('LCH normalized:', result['lch_norm'])
    print('WUP:', result['wup'])
    print('Lin:', result['lin'])

9 content words
18 comparable pairs
18 noun/verb pairs skipped

man girl
LCS: ['adult.n.01']
Path: 0.25
LCH: 2.2512917986064953
LCH normalized: 0.6188971751464533
WUP: 0.631578947368421
Lin: 0.7135111237276783

man boy
LCS: ['male.n.02']
Path: 0.3333333333333333
LCH: 2.538973871058276
LCH normalized: 0.6979831568441128
WUP: 0.6666666666666666
Lin: 0.7294717876200584

man woman
LCS: ['adult.n.01']
Path: 0.3333333333333333
LCH: 2.538973871058276
LCH normalized: 0.6979831568441128
WUP: 0.6666666666666666
Lin: 0.7870841372982784

man dog
LCS: ['organism.n.01']
Path: 0.14285714285714285
LCH: 1.6916760106710724
LCH normalized: 0.4650545544186692
WUP: 0.6666666666666666
Lin: 0.29215730428718933

man cat
LCS: ['organism.n.01']
Path: 0.08333333333333333
LCH: 1.1526795099383855
LCH normalized: 0.3168803319905661
WUP: 0.5217391304347826
Lin: 0.2914942120509087

swim walk
LCS: ['travel.v.01']
Path: 0.3333333333333333
LCH: 2.159484249353372
LCH normalized: 0.6628054829415044
WUP: 0.3333333333333333

*What similarity seems better?*